# Gene Expression Heatmaps — UPS Project (31 and 26 Samples)

## Description

This notebook creates gene expression heatmaps for 31 RNA-seq samples and for 26 samples after removing five samples.

It compares sample clustering using Pearson, cosine, and Euclidean distances. Tumor types are shown below the heatmaps.

## Requirements

### Input Data

- `~/Documents/ups/data/ps_31sample.vst.tsv`: Normalized gene expression data (VST) for 31 samples.
- `~/Documents/ups/data/methylation/methyl_classifications.tsv`: Sample tumor-type information.

### Required Software

- R with Jupyter Notebook (R kernel)
- R packages: `dplyr`, `tibble`, `readr`, `readxl`, `grid`, `ComplexHeatmap`, `extrafont`, `svglite`

### Settings

- Five samples removed for the 26-sample analysis: `PDXSAR010`, `PDXSAR023`, `SAR023`, `SAR050`, and `SAR009`.
- Distance methods: Pearson, cosine, and Euclidean.

## Results

### 1. What important information do I gain from this analysis?

- See how samples cluster based on gene expression.
- Compare sample clustering before and after removing five samples, using three distance methods.

### 2. What plots, files etc. are created and where?

- Six heatmaps: one for each distance method (Pearson, cosine, and Euclidean) for both 31 and 26 samples.
- Saved as PNG (300 dpi) and SVG (12 × 10 inches).

**Output directory:** `/Users/ziyao/Documents/ups/analyses/rnaseq/results/`

In [ ]:
Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer

suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tibble))
suppressPackageStartupMessages(library(readr))
suppressPackageStartupMessages(library(readxl))
suppressPackageStartupMessages(library(grid))
suppressPackageStartupMessages(library(ComplexHeatmap))
suppressPackageStartupMessages(library(extrafont))
suppressPackageStartupMessages(library(svglite))

getwd()
sessionInfo()

In [ ]:
# Specify global plotting settings
suppressMessages(extrafont::font_import(pattern="Arial",prompt=FALSE))
suppressMessages(extrafont::loadfonts())
gpmaj = grid::gpar(fontsize = 14, fontfamily = "Arial")
gpmin = grid::gpar(fontsize = 10, fontfamily = "Arial")

options(repr.plot.width = 12, repr.plot.height = 10)

In [ ]:
# Useful functions
save_heatmap <- function(hm, filename_base, width = 8, height = 8, res = 300) {
  # Save as PNG
  png(paste0(filename_base, ".png"), width = width, height = height, units = 'in', res = res)
  draw(hm)
  dev.off()
  
  # Save as SVG
  svglite(paste0(filename_base, ".svg"), width = width, height = height)
  draw(hm)
  dev.off()
}

In [ ]:
# Load data
read_rna_exp <- function(file){
    data = read.table(file,header=TRUE,sep='\t',row.names=1) %>%
        as.matrix
    return(data)
}
normalized_rna_path = '~/Documents/ups/data/ps_31sample.vst.tsv'
normalized_rna_data = read_rna_exp(normalized_rna_path)
normalized_rna_data %>% dim

read_metadata <- function(metadata_location,sample_set = NULL){
    annot = read.table(metadata_location, check.names=FALSE,row.names=1,header=TRUE)
    rownames(annot) <- gsub("_.*$", "", rownames(annot)) # drop suffixes
    if (!is.null(sample_set)){
        annot = annot[rownames(annot) %in% sample_set, ]
        annot[setdiff(sample_set,rownames(annot)), ] = NA
        annot = annot[sample_set, ]
    }
    return(annot)
}
metadata_path = '~/Documents/ups/data/methylation/methyl_classifications.tsv'
metadata = read_metadata(metadata_path, sample_set = colnames(normalized_rna_data))
metadata %>% head

In [ ]:
# Correct the class label
metadata$class[metadata$class == "UPS/MFS/LS_PL/RMS_PL?"] <- "UPS/MFS/LS_PL/RMS_PL"

# Check the number of samples in each class
table(metadata$class, useNA = "ifany")

In [ ]:
# Check the dimensions of the expression matrix
dim(normalized_rna_data)

In [ ]:
# Define colors for each sample class
class_colors <- c(
    `UPS/MFS/LS_PL/RMS_PL`='chocolate',
    CTRL_BONE='ivory',CTRL_MUS='mistyrose',
    HS='steelblue',
    MPNST='cyan',
    NC='gray30',
    DDLS='seagreen'
)

# Create heatmap
make_heatmap <- function(normalized_rna_data,metadata,clustering_distance_columns){
     # Add tumor type annotation
    a  <- HeatmapAnnotation(
        `tumor type` = metadata$class,
        col= list(`tumor type`=class_colors),
        which='column'
    )
    hm_31_pearson <- Heatmap(normalized_rna_data,
        clustering_distance_columns=clustering_distance_columns,
        clustering_distance_rows='euclidean',
        show_row_names=FALSE,
        row_dend_width = unit(0.25, "npc"),
        show_row_dend=FALSE,
        column_dend_height = unit(0.15, "npc"),
        heatmap_legend_param = list(title = 'gene\nexpression'),
        bottom_annotation = a
    )
}
# Generate heatmap using Pearson distance
hm_31_pearson <- make_heatmap(normalized_rna_data,metadata,'pearson')

In [ ]:
# Save heatmap
save_heatmap(
    hm_31_pearson,
    "/Users/ziyao/Documents/ups/analyses/rnaseq/results/260819heatmap_31samples_pearson",
    width = 12,
    height = 10
)

In [ ]:
# Exclude 5 samples to create the 26-sample dataset
exclude_samples <- c(
    "PDXSAR010",
    "PDXSAR023",
    "SAR023",
    "SAR050",
    "SAR009"
)

normalized_rna_data_26 <- normalized_rna_data[
    ,
    !(colnames(normalized_rna_data) %in% exclude_samples)
]

# Match metadata order to the 26-sample expression matrix
metadata_26 <- metadata[
    colnames(normalized_rna_data_26),
    ,
    drop = FALSE
]

In [ ]:
# Check the 26-sample dataset
dim(normalized_rna_data_26)
dim(metadata_26)
all(colnames(normalized_rna_data_26) == rownames(metadata_26))

In [ ]:
# Generate heatmap for 26 samples using Pearson distance
hm <- make_heatmap(normalized_rna_data_26,metadata_26,'pearson')

In [ ]:
# Save 26-sample Pearson heatmap
save_heatmap(
    hm,
    "/Users/ziyao/Documents/ups/analyses/rnaseq/results/heatmap_26samples_pearson",
    width = 12,
    height = 10
)

# Cosine distance

In [ ]:
# Define cosine distance
cosine_dist <- function(m) {
    m_norm <- m / sqrt(rowSums(m^2))
    as.dist(1 - tcrossprod(m_norm))
}
# Generate cosine distance heatmaps
hm_26_cosine <- make_heatmap(normalized_rna_data_26,metadata_26,cosine_dist)
hm_31_cosine <- make_heatmap(normalized_rna_data,metadata,cosine_dist)

In [ ]:
draw(hm_26_cosine)

In [ ]:
# Save 26-sample cosine heatmap
save_heatmap(
    hm_26_cosine,
    "/Users/ziyao/Documents/ups/analyses/rnaseq/results/heatmap_26samples_cosine",
    width = 12,
    height = 10
)

In [ ]:
# Save 31-sample cosine heatmap
save_heatmap(
    hm_31_cosine,
    "/Users/ziyao/Documents/ups/analyses/rnaseq/results/heatmap_31samples_cosine",
    width = 12,
    height = 10
)

# Euclidean distance

In [ ]:
# Generate Euclidean distance heatmaps
hm_26_euclidean <- make_heatmap(normalized_rna_data_26,metadata_26,'euclidean')
hm_31_euclidean <- make_heatmap(normalized_rna_data,metadata,'euclidean')

In [ ]:
draw(hm_26_euclidean)

In [ ]:
# Save 26-sample Euclidean heatmap
save_heatmap(
    hm_26_euclidean,
    "/Users/ziyao/Documents/ups/analyses/rnaseq/results/heatmap_26samples_euclidean",
    width = 12,
    height = 10
)

In [ ]:
# Save 31-sample Euclidean heatmap
save_heatmap(
    hm_31_euclidean,
    "/Users/ziyao/Documents/ups/analyses/rnaseq/results/heatmap_31samples_euclidean",
    width = 12,
    height = 10
)